<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/NuscenesEval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader

TORCH_version = torch.__version__.split('+')[0]
#CUDA_version = torch.version.cuda.replace('.', '')

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

#import torch_sparse

from scipy.ndimage import maximum_filter
from scipy.spatial._qhull import ConvexHull

import sys

#!pip install import-ipynb
#import import_ipynb

#!pip install open3d plotly
#import open3d as o3d

import matplotlib.pyplot as plt
import json

Mounted at /content/drive


In [ ]:
!npx degit google-research-datasets/Objectron/objectron objectron

!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

In [ ]:
sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box as BoxIoU

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box
from nuscenes.eval.detection.evaluate import DetectionEval
from nuscenes.eval.detection.data_classes import DetectionConfig
from nuscenes.eval.detection.config import config_factory

from pyquaternion import Quaternion

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

In [ ]:
sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

import lidartrainlibrary as ltb

In [ ]:
def label_to_category(k):
  categories = ["barrier", "bicycle", "bus", "car", "construction_vehicle", "motorcycle", "pedestrian", "traffic_cone", "trailer", "truck"]

  return categories[k]

In [ ]:
#[l, w, h, sin, cos, x,y,z ,k, I]

def results_to_json(nusc, boxes_per_sample, sample_tokens, out_path, use_camera = True): #add val key to model, which allows it to output sample tokens

  results = {}
  for i in range(len(sample_tokens)):
    formatted = []
    for box in boxes_per_sample[i]:

      sample = nusc.get('sample', sample_tokens[i])

      ego_pose = nusc.get("ego_pose", sample["data"]["LIDAR_TOP"])
      sd_rec = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
      cal_sensor = nusc.get('calibrated_sensor', sd_rec['calibrated_sensor_token'])

      box_size = np.exp(np.array([box[0], box[1], box[2]]))
      raw_translation = np.array([box[5], box[6], box[7]])

      rot_matrix = np.array([[box[4],-box[3], 0],[box[3],box[4],0],[0,0,1]])
      raw_rotation = Quaternion(matrix = rot_matrix)

      translation = raw_translation + np.array(ego_pose["translation"]) + np.array(cal_sensor["translation"])
      rotation = Quaternion(cal_sensor["translation"]) * Quaternion(ego_pose["rotation"]) * raw_rotation

      box_dict = {
          "sample": sample_tokens[i],
          "size": box_size,
          "translation": translation,
          "rotation": rotation,
          "velocity": [0.0, 0.0],
          "detection_name": label_to_category(box[8]),
          "detection_score": box[9],
          "attribute_name": ""
      }

      formatted.append(box_dict)

    results[sample_tokens[i]] = formatted

  submission = {
    "meta": {
        "use_camera": use_camera,     # set True/False for baseline vs fusion
        "use_lidar": True,
        "use_radar": False,
        "use_map": False,
        "use_external": False,
    },
    "results": results,
  }

  with open(out_path, 'w') as f:
    json.dump(submission, f)

  return out_path

In [3]:
def run_eval(result_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    cfg = config_factory('detection_cvpr_2019')  # standard nuScenes detection config

    nusc_eval = DetectionEval(
        nusc,
        config=cfg,
        result_path=result_path,
        eval_set='mini_val',
        output_dir=output_dir,
        verbose=True,
    )
    metrics_summary = nusc_eval.main(plot_examples=0, render_curves=False)

    return metrics_summary
